# Repeat Modeling
Some inversions (like the ones in the Medium class) are simply **never** found in the linked-read data. So, it begs the question of whether there's something notable about the genomic regions they happen to be in-- the first suspicion is that they're in repeat regions and the sequences failed to align well there. These same Medium inversions were picked up reasonably well with the simulated long-read data, which further fuels this suspicion.

### Getting the repeat database
Cornell has a RepBase license and [specific instructions](https://biohpc.cornell.edu/lab/userguide.aspx?a=software&i=259#c) on how to download a copy of the database locally. After retrieving that file, we need to rebuild the database.

In [2]:
%%bash
tar -xvf RepBase30.09.embl.tar.gz

RepBase30.09.embl/
RepBase30.09.embl/appendix/
RepBase30.09.embl/appendix/angapp.ref
RepBase30.09.embl/appendix/athapp.ref
RepBase30.09.embl/appendix/bctapp.ref
RepBase30.09.embl/appendix/cbrapp.ref
RepBase30.09.embl/appendix/celapp.ref
RepBase30.09.embl/appendix/chlapp.ref
RepBase30.09.embl/appendix/cinapp.ref
RepBase30.09.embl/appendix/diaapp.ref
RepBase30.09.embl/appendix/droapp.ref
RepBase30.09.embl/appendix/edcotapp.ref
RepBase30.09.embl/appendix/fngapp.ref
RepBase30.09.embl/appendix/fugapp.ref
RepBase30.09.embl/appendix/grasapp.ref
RepBase30.09.embl/appendix/humapp.ref
RepBase30.09.embl/appendix/invapp.ref
RepBase30.09.embl/appendix/mamapp.ref
RepBase30.09.embl/appendix/mcotapp.ref
RepBase30.09.embl/appendix/nemapp.ref
RepBase30.09.embl/appendix/oryapp.ref
RepBase30.09.embl/appendix/plnapp.ref
RepBase30.09.embl/appendix/priapp.ref
RepBase30.09.embl/appendix/rodapp.ref
RepBase30.09.embl/appendix/spuapp.ref
RepBase30.09.embl/appendix/synapp.ref
RepBase30.09.embl/appendix/unaapp.ref

From the list of possible reference databases, we want `drorep.ref`, which is the Drosophila one.

In [ ]:
%%bash
/programs/bin/perlscripts/embl2rm.pl RepBase30.09.embl/drorep.ref dmel.lib

Still following the instructions provided by Cornell BioHPC, we will create our own local Drosophila repeat database

In [11]:
%%bash 
## create your own copy of RepeatMasker database 
singularity run --bind $PWD --pwd $PWD ./tetools.sif cp -r /opt/RepeatMasker ./

## delete the built in databae
cd RepeatMasker 
rm Libraries/RepeatMasker.lib*

## replace "myDatabase.lib" with the .lib file we produced from previous step 
cp ../dmel.lib Libraries/RepeatMasker.lib
/programs/bin/blast+/makeblastdb -dbtype nucl -in Libraries/RepeatMasker.lib



Building a new DB, current time: 09/29/2025 11:46:38
New DB name:   /local/workdir/pd348/haplotagging_simulations/dmel_genome/RepeatMasker/Libraries/RepeatMasker.lib
New DB title:  Libraries/RepeatMasker.lib
Sequence type: Nucleotide
Keep MBits: T
Maximum file size: 3000000000B
Adding sequences from FASTA; added 327 sequences in 0.0135379 seconds.




First, we will do a basic classification. After that, we will re-do the classification with the custom Drosophila database

In [ ]:
%%bash 

#build database
singularity run --bind $PWD --pwd $PWD ./tetools.sif BuildDatabase -name dmel dmel_2_3.fa

#run RepeatModeler
singularity run --bind $PWD --pwd $PWD ./tetools.sif RepeatModeler -database dmel -threads 20 -LTRStruct 

Re-run repeatClassifier with the custom Drosophila database

- when you run RepeatClassifier, copy over the RepeatMasker directory to same place as the consensi.fa file. 
- specify your RepeatMasker directory which include the custom db `-repeatmasker_dir ./RepeatMasker`

In [3]:
%%bash

singularity run --bind $PWD --pwd $PWD ./tetools.sif RepeatClassifier -consensi dmel_2_3.fa -repeatmasker_dir ./RepeatMasker > new_consensi.fa.classified

NCBIBlastSearchEngine::search: Error...matrix (/opt/RepeatModeler/Matrices/ncbi/nt/comparison.matrix) does not exist!
 at /opt/RepeatModeler/RepeatClassifier line 497.


Process is interrupted.
